# FSQR

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "fsrq_sed_dark_theme.gif"

FPS = 24
DURATION_SEC = 8
FRAMES = FPS * DURATION_SEC

FIGSIZE = (16, 9)
DPI = 120

BG = "#030711"
GRID = "#16324a"
TEXT = "#c9d3df"
MUTED = "#6f8194"

CYAN = "#35c9ff"
BLUE_FILL = "#0b63ff"
ORANGE = "#ff6b32"
RED_FILL = "#ff3b1f"
PURPLE = "#b76cff"

# =========================
# Data: schematic FSRQ SED
# =========================

x = np.linspace(8.3, 25.2, 1200)  # log10 frequency

def log_bump(x, center, height, width):
    return height * np.exp(-0.5 * ((x - center) / width) ** 2)

# Two-hump SED in log space
syn = log_bump(x, center=14.1, height=1.00, width=1.55)
ic = log_bump(x, center=21.5, height=1.18, width=1.35)

# Smooth valley between components
baseline = 0.035
total = baseline + syn + ic

# Thermal UV bump / accretion disk
disk = baseline + 0.34 * np.exp(-0.5 * ((x - 15.0) / 0.72) ** 2)

# Mask disk outside visual range
disk_masked = np.where((x > 12.1) & (x < 16.7), disk, np.nan)

# =========================
# Figure setup
# =========================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

ax.set_xlim(8.2, 25.5)
ax.set_ylim(0, 1.45)

for spine in ax.spines.values():
    spine.set_color("#95a3b5")
    spine.set_linewidth(1.2)

ax.tick_params(colors=TEXT, labelsize=12, length=6)

ax.set_xticks([10, 12, 14, 16, 18, 20, 22, 24])
ax.set_xticklabels([r"$10^{10}$", r"$10^{12}$", r"$10^{14}$", r"$10^{16}$",
                    r"$10^{18}$", r"$10^{20}$", r"$10^{22}$", r"$10^{24}$"])

ax.set_yticks([0.2, 0.45, 0.7, 0.95, 1.2])
ax.set_yticklabels([r"$10^{-17}$", r"$10^{-15}$", r"$10^{-13}$", r"$10^{-11}$", r"$10^{-9}$"])

ax.grid(True, which="major", color=GRID, linestyle=":", linewidth=1.1, alpha=0.75)

ax.set_xlabel(r"$\nu$  (Hz)", color=TEXT, fontsize=18, labelpad=14)
ax.set_ylabel(r"$\nu F_{\nu}$", color=TEXT, fontsize=20, labelpad=12)

# Titles
fig.text(0.5, 0.94, "FSRQ QUASAR — SPECTRAL ENERGY DISTRIBUTION",
         ha="center", va="center", color="#d7dde8", fontsize=24, fontweight="bold")

fig.text(0.5, 0.90, "SCHEMATIC TWO-HUMP SED",
         ha="center", va="center", color=MUTED, fontsize=15, fontweight="bold")

# Region labels
ax.text(13.8, 1.30, "SYNCHROTRON\nEMISSION",
        color=CYAN, fontsize=17, fontweight="bold", ha="center")

ax.text(21.5, 1.30, "INVERSE COMPTON\nEMISSION (EC DOMINANT)",
        color=ORANGE, fontsize=17, fontweight="bold", ha="center")

# Band labels
bands = [
    (10.0, "RADIO", "⌁"),
    (12.2, "IR", "✺"),
    (14.3, "OPTICAL", "◉"),
    (16.5, "UV", "☼"),
    (18.7, "X-RAY", "✶"),
    (22.0, "GAMMA-RAY", "γ"),
]

for bx, label, icon in bands:
    col = CYAN if bx < 16 else (PURPLE if bx < 20 else ORANGE)
    ax.text(bx, 0.115, label, color=col, fontsize=13, fontweight="bold", ha="center")
    ax.text(bx, 0.055, icon, color=col, fontsize=25, ha="center", va="center")

# Static annotations
ax.text(15.0, 0.46, "ACCRETION\nDISK", color=PURPLE,
        fontsize=14, fontweight="bold", ha="center")

ax.text(24.75, 0.03, "NOT TO SCALE — SCHEMATIC ONLY",
        color="#415064", fontsize=9, ha="right")

# Bottom explanatory strip
ax.text(11.4, -0.16, "LOW ENERGY", transform=ax.get_xaxis_transform(),
        color=CYAN, fontsize=12, ha="center", fontweight="bold")

ax.text(21.6, -0.16, "HIGH ENERGY", transform=ax.get_xaxis_transform(),
        color=ORANGE, fontsize=12, ha="center", fontweight="bold")

ax.text(17.0, -0.25, "SEED PHOTONS FOR IC SCATTERING",
        transform=ax.get_xaxis_transform(),
        color=TEXT, fontsize=11, ha="center",
        bbox=dict(boxstyle="round,pad=0.35", facecolor="#07111f", edgecolor="#2b4358"))

# =========================
# Animated artists
# =========================

syn_fill = ax.fill_between([], [], [], color=BLUE_FILL, alpha=0.0)
ic_fill = ax.fill_between([], [], [], color=RED_FILL, alpha=0.0)

# Glow layers
syn_glow1, = ax.plot([], [], color=CYAN, linewidth=9, alpha=0.10)
syn_glow2, = ax.plot([], [], color=CYAN, linewidth=5, alpha=0.18)

ic_glow1, = ax.plot([], [], color=ORANGE, linewidth=9, alpha=0.10)
ic_glow2, = ax.plot([], [], color=ORANGE, linewidth=5, alpha=0.18)

# Main colored curves
syn_line, = ax.plot([], [], color=CYAN, linewidth=2.9)
ic_line, = ax.plot([], [], color=ORANGE, linewidth=2.9)

disk_line, = ax.plot(
    [], [],
    color=PURPLE,
    linewidth=2.0,
    linestyle=(0, (4, 4)),
    alpha=0.95
)

cursor, = ax.plot([], [], marker="o", markersize=7, color="#ffffff", alpha=0.95)


# =========================
# Animation functions
# =========================

def ease(t):
    return 1 - (1 - t) ** 3


def update(frame):
    global syn_fill, ic_fill

    t = ease(frame / (FRAMES - 1))
    xmax = x.min() + t * (x.max() - x.min())

    visible = x <= xmax

    syn_part = visible & (x <= 17.8)
    ic_part = visible & (x >= 17.3)

    # Synchrotron curve
    syn_line.set_data(x[syn_part], total[syn_part])
    syn_glow1.set_data(x[syn_part], total[syn_part])
    syn_glow2.set_data(x[syn_part], total[syn_part])

    # Inverse Compton curve
    ic_line.set_data(x[ic_part], total[ic_part])
    ic_glow1.set_data(x[ic_part], total[ic_part])
    ic_glow2.set_data(x[ic_part], total[ic_part])

    # Cursor follows the full currently visible SED
    if np.any(visible):
        xv = x[visible]
        yv = total[visible]
        cursor.set_data([xv[-1]], [yv[-1]])

    # Remove and redraw fills
    syn_fill.remove()
    ic_fill.remove()

    syn_fill = ax.fill_between(
        x[syn_part],
        baseline,
        total[syn_part],
        color=BLUE_FILL,
        alpha=0.20
    )

    ic_fill = ax.fill_between(
        x[ic_part],
        baseline,
        total[ic_part],
        color=RED_FILL,
        alpha=0.18
    )

    # Accretion disk appears in UV/optical region
    disk_visible = visible & np.isfinite(disk_masked)
    disk_line.set_data(x[disk_visible], disk_masked[disk_visible])

    return (
        syn_line, syn_glow1, syn_glow2,
        ic_line, ic_glow1, ic_glow2,
        cursor, disk_line,
        syn_fill, ic_fill
    )

anim = FuncAnimation(fig, update, frames=FRAMES, interval=1000 / FPS, blit=False)

# =========================
# Save GIF
# =========================

writer = PillowWriter(fps=FPS)
anim.save(GIF_PATH, writer=writer)

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print(f"Saved: {GIF_PATH}")

# BL LAC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "bl_lac_sed_dark_theme.gif"

FPS = 24
DURATION_SEC = 8
FRAMES = FPS * DURATION_SEC

FIGSIZE = (16, 9)
DPI = 120

BG = "#030711"
GRID = "#16324a"
TEXT = "#c9d3df"
MUTED = "#6f8194"

CYAN = "#35c9ff"
BLUE_FILL = "#0b63ff"
ORANGE = "#ff6b32"
RED_FILL = "#ff3b1f"
PURPLE = "#b76cff"

# =========================
# Data: schematic BL Lac SED
# =========================

x = np.linspace(8.3, 25.2, 1200)  # log10 frequency

def log_bump(x, center, height, width):
    return height * np.exp(-0.5 * ((x - center) / width) ** 2)

# BL Lac: non-thermal continuum, weak/absent lines,
# synchrotron peak shifted higher than in FSRQ.
baseline = 0.035

syn = log_bump(x, center=15.6, height=1.05, width=1.65)
ic = log_bump(x, center=23.0, height=0.92, width=1.45)

total = baseline + syn + ic

# Weak host galaxy / very weak thermal contribution, only as faint optional trace
host = baseline + 0.10 * np.exp(-0.5 * ((x - 14.4) / 0.85) ** 2)
host_masked = np.where((x > 12.2) & (x < 16.4), host, np.nan)

# =========================
# Figure setup
# =========================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

ax.set_xlim(8.2, 25.5)
ax.set_ylim(0, 1.45)

for spine in ax.spines.values():
    spine.set_color("#95a3b5")
    spine.set_linewidth(1.2)

ax.tick_params(colors=TEXT, labelsize=12, length=6)

ax.set_xticks([10, 12, 14, 16, 18, 20, 22, 24])
ax.set_xticklabels([
    r"$10^{10}$", r"$10^{12}$", r"$10^{14}$", r"$10^{16}$",
    r"$10^{18}$", r"$10^{20}$", r"$10^{22}$", r"$10^{24}$"
])

ax.set_yticks([0.2, 0.45, 0.7, 0.95, 1.2])
ax.set_yticklabels([
    r"$10^{-17}$", r"$10^{-15}$", r"$10^{-13}$",
    r"$10^{-11}$", r"$10^{-9}$"
])

ax.grid(True, which="major", color=GRID, linestyle=":", linewidth=1.1, alpha=0.75)

ax.set_xlabel(r"$\nu$  (Hz)", color=TEXT, fontsize=18, labelpad=14)
ax.set_ylabel(r"$\nu F_{\nu}$", color=TEXT, fontsize=20, labelpad=12)

# Titles
fig.text(
    0.5, 0.94,
    "BL LACERTAE OBJECT — SPECTRAL ENERGY DISTRIBUTION",
    ha="center", va="center",
    color="#d7dde8", fontsize=23, fontweight="bold"
)

fig.text(
    0.5, 0.90,
    "SCHEMATIC TWO-HUMP NON-THERMAL SED",
    ha="center", va="center",
    color=MUTED, fontsize=15, fontweight="bold"
)

# Region labels
ax.text(
    15.3, 1.30,
    "SYNCHROTRON\nEMISSION",
    color=CYAN, fontsize=17, fontweight="bold", ha="center"
)

ax.text(
    22.7, 1.30,
    "INVERSE COMPTON\nEMISSION (SSC DOMINANT)",
    color=ORANGE, fontsize=17, fontweight="bold", ha="center"
)

# Band labels
bands = [
    (10.0, "RADIO", "⌁"),
    (12.2, "IR", "✺"),
    (14.3, "OPTICAL", "◉"),
    (16.5, "UV", "☼"),
    (18.7, "X-RAY", "✶"),
    (22.0, "GAMMA-RAY", "γ"),
]

for bx, label, icon in bands:
    col = CYAN if bx < 17 else (PURPLE if bx < 20 else ORANGE)
    ax.text(bx, 0.115, label, color=col, fontsize=13, fontweight="bold", ha="center")
    ax.text(bx, 0.055, icon, color=col, fontsize=25, ha="center", va="center")

# BL Lac notes
ax.text(
    14.3, 0.28,
    "WEAK / ABSENT\nTHERMAL FEATURES",
    color=PURPLE,
    fontsize=12,
    fontweight="bold",
    ha="center",
    alpha=0.9
)

ax.text(
    24.75, 0.03,
    "NOT TO SCALE — SCHEMATIC ONLY",
    color="#415064", fontsize=9, ha="right"
)

# Bottom explanatory strip
ax.text(
    11.4, -0.16,
    "LOW ENERGY",
    transform=ax.get_xaxis_transform(),
    color=CYAN, fontsize=12, ha="center", fontweight="bold"
)

ax.text(
    21.8, -0.16,
    "HIGH ENERGY",
    transform=ax.get_xaxis_transform(),
    color=ORANGE, fontsize=12, ha="center", fontweight="bold"
)

ax.text(
    17.3, -0.25,
    "JET-DOMINATED CONTINUUM — WEAK EMISSION LINES",
    transform=ax.get_xaxis_transform(),
    color=TEXT, fontsize=11, ha="center",
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="#07111f",
        edgecolor="#2b4358"
    )
)

# =========================
# Animated artists
# =========================

syn_fill = ax.fill_between([], [], [], color=BLUE_FILL, alpha=0.0)
ic_fill = ax.fill_between([], [], [], color=RED_FILL, alpha=0.0)

syn_glow1, = ax.plot([], [], color=CYAN, linewidth=9, alpha=0.10)
syn_glow2, = ax.plot([], [], color=CYAN, linewidth=5, alpha=0.18)

ic_glow1, = ax.plot([], [], color=ORANGE, linewidth=9, alpha=0.10)
ic_glow2, = ax.plot([], [], color=ORANGE, linewidth=5, alpha=0.18)

syn_line, = ax.plot([], [], color=CYAN, linewidth=2.9)
ic_line, = ax.plot([], [], color=ORANGE, linewidth=2.9)

host_line, = ax.plot(
    [], [],
    color=PURPLE,
    linewidth=1.8,
    linestyle=(0, (3, 5)),
    alpha=0.55
)

cursor, = ax.plot([], [], marker="o", markersize=7, color="#ffffff", alpha=0.95)

# =========================
# Animation functions
# =========================

def ease(t):
    return 1 - (1 - t) ** 3


def update(frame):
    global syn_fill, ic_fill

    t = ease(frame / (FRAMES - 1))
    xmax = x.min() + t * (x.max() - x.min())

    visible = x <= xmax

    syn_part = visible & (x <= 18.6)
    ic_part = visible & (x >= 18.0)

    syn_line.set_data(x[syn_part], total[syn_part])
    syn_glow1.set_data(x[syn_part], total[syn_part])
    syn_glow2.set_data(x[syn_part], total[syn_part])

    ic_line.set_data(x[ic_part], total[ic_part])
    ic_glow1.set_data(x[ic_part], total[ic_part])
    ic_glow2.set_data(x[ic_part], total[ic_part])

    if np.any(visible):
        xv = x[visible]
        yv = total[visible]
        cursor.set_data([xv[-1]], [yv[-1]])

    syn_fill.remove()
    ic_fill.remove()

    syn_fill = ax.fill_between(
        x[syn_part],
        baseline,
        total[syn_part],
        color=BLUE_FILL,
        alpha=0.20
    )

    ic_fill = ax.fill_between(
        x[ic_part],
        baseline,
        total[ic_part],
        color=RED_FILL,
        alpha=0.18
    )

    host_visible = visible & np.isfinite(host_masked)
    host_line.set_data(x[host_visible], host_masked[host_visible])

    return (
        syn_line, syn_glow1, syn_glow2,
        ic_line, ic_glow1, ic_glow2,
        cursor, host_line,
        syn_fill, ic_fill
    )

anim = FuncAnimation(fig, update, frames=FRAMES, interval=1000 / FPS, blit=False)

# =========================
# Save GIF
# =========================

writer = PillowWriter(fps=FPS)
anim.save(GIF_PATH, writer=writer)

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print(f"Saved: {GIF_PATH}")

# Spectra

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "fsrq_vs_bl_lac_optical_lines.gif"

FPS = 24
DURATION_SEC = 8
FRAMES = FPS * DURATION_SEC

FIGSIZE = (16, 9)
DPI = 120

BG = "#030711"
GRID = "#16324a"
TEXT = "#c9d3df"
MUTED = "#6f8194"

CYAN = "#35c9ff"
ORANGE = "#ff6b32"
PURPLE = "#b76cff"
GREEN = "#48ffb3"

# =========================
# Data: optical spectrum schematic
# =========================

wl = np.linspace(3500, 7500, 1600)  # Angstrom

def gaussian(x, mu, amp, sigma):
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

# Smooth continua
fsrq_cont = 0.55 + 0.18 * (wl / 5500) ** -0.8
bl_cont = 0.72 + 0.05 * np.sin((wl - 3500) / 900)

# FSRQ broad emission lines
fsrq = fsrq_cont.copy()
fsrq += gaussian(wl, 4861, 0.55, 85)    # H-beta
fsrq += gaussian(wl, 5007, 0.32, 55)    # [O III]
fsrq += gaussian(wl, 6563, 0.70, 110)   # H-alpha
fsrq += gaussian(wl, 4340, 0.25, 75)    # H-gamma
fsrq += gaussian(wl, 3727, 0.22, 65)    # [O II]

# BL Lac: jet continuum, weak/absent lines
bl_lac = bl_cont.copy()
bl_lac += gaussian(wl, 4861, 0.045, 45)
bl_lac += gaussian(wl, 6563, 0.035, 60)

# =========================
# Figure setup
# =========================

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=FIGSIZE,
    dpi=DPI,
    sharex=True,
    gridspec_kw={"height_ratios": [1, 1], "hspace": 0.14}
)

fig.patch.set_facecolor(BG)

for ax in (ax1, ax2):
    ax.set_facecolor(BG)
    ax.set_xlim(3500, 7500)
    ax.set_ylim(0.35, 1.55)
    ax.grid(True, color=GRID, linestyle=":", linewidth=1.0, alpha=0.75)
    ax.tick_params(colors=TEXT, labelsize=11)

    for spine in ax.spines.values():
        spine.set_color("#95a3b5")
        spine.set_linewidth(1.1)

ax2.set_xlabel("Wavelength  λ  (Å)", color=TEXT, fontsize=17, labelpad=12)
ax1.set_ylabel("Relative flux", color=TEXT, fontsize=15)
ax2.set_ylabel("Relative flux", color=TEXT, fontsize=15)

fig.text(
    0.5, 0.955,
    "OPTICAL SPECTRA: FSRQ vs BL LAC",
    ha="center", va="center",
    color="#d7dde8",
    fontsize=24,
    fontweight="bold"
)

fig.text(
    0.5, 0.918,
    "Emission lines are visible in optical spectra, not in broad-band SED plots",
    ha="center", va="center",
    color=MUTED,
    fontsize=14,
    fontweight="bold"
)

ax1.text(
    0.02, 0.86,
    "FSRQ — strong broad emission lines",
    transform=ax1.transAxes,
    color=ORANGE,
    fontsize=17,
    fontweight="bold"
)

ax2.text(
    0.02, 0.86,
    "BL Lac — weak or absent emission lines",
    transform=ax2.transAxes,
    color=CYAN,
    fontsize=17,
    fontweight="bold"
)

# Line labels
line_labels = [
    (3727, "[O II]"),
    (4340, "Hγ"),
    (4861, "Hβ"),
    (5007, "[O III]"),
    (6563, "Hα"),
]

for x0, label in line_labels:
    ax1.axvline(x0, color=PURPLE, alpha=0.20, linewidth=1.2)
    ax2.axvline(x0, color=PURPLE, alpha=0.10, linewidth=1.0)

    ax1.text(
        x0, 1.48,
        label,
        color=PURPLE,
        fontsize=11,
        ha="center",
        va="top",
        fontweight="bold"
    )

ax1.text(
    0.72, 0.14,
    "broad-line region visible",
    transform=ax1.transAxes,
    color=GREEN,
    fontsize=13,
    fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.35", facecolor="#07111f", edgecolor="#2b4358")
)

ax2.text(
    0.65, 0.14,
    "jet continuum overwhelms weak features",
    transform=ax2.transAxes,
    color=CYAN,
    fontsize=13,
    fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.35", facecolor="#07111f", edgecolor="#2b4358")
)

# =========================
# Animated artists
# =========================

fsrq_glow1, = ax1.plot([], [], color=ORANGE, linewidth=9, alpha=0.10)
fsrq_glow2, = ax1.plot([], [], color=ORANGE, linewidth=5, alpha=0.18)
fsrq_line, = ax1.plot([], [], color=ORANGE, linewidth=2.8)

bl_glow1, = ax2.plot([], [], color=CYAN, linewidth=9, alpha=0.10)
bl_glow2, = ax2.plot([], [], color=CYAN, linewidth=5, alpha=0.18)
bl_line, = ax2.plot([], [], color=CYAN, linewidth=2.8)

cursor1, = ax1.plot([], [], marker="o", markersize=6, color="#ffffff", alpha=0.9)
cursor2, = ax2.plot([], [], marker="o", markersize=6, color="#ffffff", alpha=0.9)

# =========================
# Animation
# =========================

def ease(t):
    return 1 - (1 - t) ** 3


def update(frame):
    t = ease(frame / (FRAMES - 1))
    xmax = wl.min() + t * (wl.max() - wl.min())

    visible = wl <= xmax

    x_visible = wl[visible]

    y_fsrq = fsrq[visible]
    y_bl = bl_lac[visible]

    fsrq_line.set_data(x_visible, y_fsrq)
    fsrq_glow1.set_data(x_visible, y_fsrq)
    fsrq_glow2.set_data(x_visible, y_fsrq)

    bl_line.set_data(x_visible, y_bl)
    bl_glow1.set_data(x_visible, y_bl)
    bl_glow2.set_data(x_visible, y_bl)

    if len(x_visible) > 0:
        cursor1.set_data([x_visible[-1]], [y_fsrq[-1]])
        cursor2.set_data([x_visible[-1]], [y_bl[-1]])

    return (
        fsrq_line, fsrq_glow1, fsrq_glow2,
        bl_line, bl_glow1, bl_glow2,
        cursor1, cursor2
    )


anim = FuncAnimation(fig, update, frames=FRAMES, interval=1000 / FPS, blit=False)

writer = PillowWriter(fps=FPS)
anim.save(GIF_PATH, writer=writer)

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print(f"Saved: {GIF_PATH}")